# 01 - Crawl Raw Banking Data (Live)

            This notebook contains the complete crawler source used by the project. Run all cells to collect daily market data, company information, quarterly Vietstock data, the long ReportNorm audit table and annual OCR components. Live output is written only under `GENERATED_OUTPUTS/01_RAW_LIVE`; the frozen submission data is never overwritten.


In [ ]:
from pathlib import Path

def find_package_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "01_DATA_PIPELINE").is_dir() and (candidate / "02_MODELS").is_dir():
            return candidate
    if current.name == "01_DATA_PIPELINE":
        return current.parent
    raise RuntimeError("Open this notebook from inside the SUBMIT_THIS package.")

PACKAGE_ROOT = find_package_root()
PIPELINE_DIR = PACKAGE_ROOT / "01_DATA_PIPELINE"
FROZEN_RAW_DIR = PIPELINE_DIR / "FROZEN_RAW_DATA"
GENERATED_DIR = PIPELINE_DIR / "GENERATED_OUTPUTS"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
print("Package root:", PACKAGE_ROOT)


In [ ]:
import datetime as dt
import importlib.util
import os
import subprocess
import sys

END_DATE = dt.date.today().isoformat()
OUTPUT_DIR = GENERATED_DIR / "01_RAW_LIVE"
RUNTIME_SOURCE_DIR = OUTPUT_DIR / "_runtime_source"
AUTO_INSTALL_MISSING = True

required = {"pandas": "pandas", "numpy": "numpy", "requests": "requests", "vnstock": "vnstock"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError(f"Missing packages: {missing}. Install requirements.txt first.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Live crawl end date:", END_DATE)
print("Live output:", OUTPUT_DIR)


In [ ]:
RUNTIME_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_SOURCES = {'config.py': 'from __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\n\nSYMBOLS = [\n    "VCB", "BID", "CTG", "MBB", "TCB", "ACB", "VPB", "HDB", "TPB", "STB",\n    "EIB", "SHB", "VIB", "MSB", "OCB", "LPB", "SSB", "BAB", "NAB", "PGB",\n    "KLB", "VAB", "BVB", "ABB", "SGB", "NVB",\n]\n\nDEFAULT_OUTDIR = Path(r"C:\\Users\\Admin\\OneDrive\\Desktop\\bank_raw_crawl_modular_2026-05-29")\nDAILY_START = "2000-01-01"\nHF_REPO = "tinixai/ocr_annual_financials"\nHF_BRANCH = "main"\nUSER_AGENT = "Mozilla/5.0"\n\nREPORT_NORM_IDS = {\n    "roe_pct": 45,\n    "operating_profit_before_provision_million_vnd": 4376,\n    "total_assets_million_vnd": 4375,\n    "total_liabilities_and_equity_million_vnd": 4305,\n    "customer_loans_million_vnd": 4315,\n    "customer_deposits_million_vnd": 4320,\n    "equity_million_vnd": 4325,\n    "net_income_million_vnd": 4380,\n    "net_interest_income_million_vnd": 4385,\n    "provision_expense_million_vnd": 4392,\n    "reportnorm_4310_million_vnd": 4310,\n    "reportnorm_4311_million_vnd": 4311,\n    "reportnorm_4312_million_vnd": 4312,\n    "reportnorm_4313_million_vnd": 4313,\n    "reportnorm_4318_million_vnd": 4318,\n    "reportnorm_4319_million_vnd": 4319,\n    "reportnorm_4321_million_vnd": 4321,\n    "reportnorm_4322_million_vnd": 4322,\n    "reportnorm_4323_million_vnd": 4323,\n    "reportnorm_4324_million_vnd": 4324,\n}\n\nREPORT_NORM_NOTES = {\n    "reportnorm_4310_million_vnd": "liquid/current-ratio component from Vietstock ReportNormId 4310",\n    "reportnorm_4311_million_vnd": "liquid/current-ratio component from Vietstock ReportNormId 4311",\n    "reportnorm_4312_million_vnd": "liquid/current-ratio component from Vietstock ReportNormId 4312",\n    "reportnorm_4313_million_vnd": "liquid/current-ratio component from Vietstock ReportNormId 4313",\n    "reportnorm_4318_million_vnd": "short-liability/current-ratio component from Vietstock ReportNormId 4318",\n    "reportnorm_4319_million_vnd": "short-liability/current-ratio component from Vietstock ReportNormId 4319",\n    "reportnorm_4321_million_vnd": "short-liability/current-ratio component from Vietstock ReportNormId 4321",\n    "reportnorm_4322_million_vnd": "short-liability/current-ratio component from Vietstock ReportNormId 4322",\n    "reportnorm_4323_million_vnd": "short-liability/current-ratio component from Vietstock ReportNormId 4323",\n    "reportnorm_4324_million_vnd": "short-liability/current-ratio component from Vietstock ReportNormId 4324",\n}\n\n\ndef disable_proxy_env() -> None:\n    for proxy_key in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):\n        os.environ.pop(proxy_key, None)\n    os.environ["NO_PROXY"] = "*"\n    os.environ["no_proxy"] = "*"\n', 'sources_vnstock.py': 'from __future__ import annotations\n\nimport re\nfrom typing import Any\n\nimport pandas as pd\nfrom vnstock import Company, Quote\n\nfrom config import DAILY_START\n\n\ndef normalize_int(value: Any) -> int | None:\n    if value is None or value == "":\n        return None\n    stripped = re.sub(r"[^\\d]", "", str(value))\n    return int(stripped) if stripped else None\n\n\ndef fetch_daily_raw(symbol: str, end_date: str) -> tuple[pd.DataFrame, dict[str, Any]]:\n    raw = Quote(source="KBS", symbol=symbol).history(start=DAILY_START, end=end_date).copy()\n    if raw.empty:\n        raise RuntimeError(f"No daily data returned for {symbol}")\n    if "symbol" not in raw.columns:\n        raw["symbol"] = symbol\n\n    raw["time"] = pd.to_datetime(raw["time"])\n    raw = raw.sort_values("time").reset_index(drop=True)\n    raw = raw.rename(columns={"time": "trade_date", "symbol": "ticker", "close": "close_vnd"})\n\n    cols = ["trade_date", "ticker", "open", "high", "low", "close_vnd", "volume"]\n    out = raw[cols].copy()\n    out["trade_date"] = out["trade_date"].dt.strftime("%Y-%m-%d")\n\n    duplicate_mask = out.duplicated(["ticker", "trade_date"], keep=False)\n    duplicate_input_rows = int(duplicate_mask.sum())\n    conflicting_duplicate_keys = 0\n    if duplicate_input_rows:\n        value_cols = ["open", "high", "low", "close_vnd", "volume"]\n        conflicts = (\n            out.loc[duplicate_mask]\n            .groupby(["ticker", "trade_date"])[value_cols]\n            .nunique(dropna=False)\n            .gt(1)\n            .any(axis=1)\n        )\n        conflicting_duplicate_keys = int(conflicts.sum())\n        out = out.drop_duplicates(["ticker", "trade_date"], keep="last").reset_index(drop=True)\n\n    source = "vnstock.Quote(source=\'KBS\').history; deduplicated by ticker+trade_date"\n    return out, {\n        "source": source,\n        "duplicate_input_rows": duplicate_input_rows,\n        "conflicting_duplicate_keys": conflicting_duplicate_keys,\n    }\n\n\ndef fetch_company_raw(symbol: str) -> tuple[dict[str, Any], str]:\n    row = Company(source="KBS", symbol=symbol).overview().iloc[0]\n    listing_raw = row.get("listing_date")\n    listing_date = pd.to_datetime(str(listing_raw), dayfirst=True, errors="coerce")\n    normalized_listing_date = listing_date.strftime("%Y-%m-%d") if pd.notna(listing_date) else ""\n    return {\n        "ticker": symbol,\n        "listing_date_raw": listing_raw,\n        "listing_date": normalized_listing_date,\n        "outstanding_shares": normalize_int(row.get("outstanding_shares")),\n        "exchange": row.get("exchange"),\n        "as_of_date": row.get("as_of_date"),\n    }, "vnstock.Company(source=\'KBS\').overview"\n', 'sources_vietstock.py': 'from __future__ import annotations\n\nimport calendar\nimport json\nimport re\nimport time\nfrom typing import Any\n\nimport pandas as pd\nimport requests\n\nfrom config import REPORT_NORM_IDS, USER_AGENT\n\n\ndef normalize_float(value: Any) -> float | None:\n    if value is None or value == "":\n        return None\n    try:\n        if pd.isna(value):\n            return None\n    except TypeError:\n        pass\n    text = str(value).strip().replace("%", "")\n    if "," in text and "." in text:\n        if text.rfind(",") > text.rfind("."):\n            text = text.replace(".", "").replace(",", ".")\n        else:\n            text = text.replace(",", "")\n    elif text.count(".") > 1 and "," not in text:\n        text = text.replace(".", "")\n    elif text.count(",") > 1 and "." not in text:\n        text = text.replace(",", "")\n    else:\n        text = text.replace(",", "")\n    try:\n        return float(text)\n    except ValueError:\n        return None\n\n\ndef request_with_retry(session: requests.Session, method: str, url: str, **kwargs: Any) -> requests.Response:\n    headers = {"User-Agent": USER_AGENT}\n    headers.update(kwargs.pop("headers", {}) or {})\n    last_exc: Exception | None = None\n    for attempt in range(3):\n        try:\n            response = session.request(method, url, headers=headers, timeout=60, **kwargs)\n            response.raise_for_status()\n            return response\n        except Exception as exc:\n            last_exc = exc\n            time.sleep(2 + attempt * 3)\n    if last_exc:\n        raise last_exc\n    raise RuntimeError(f"Request failed: {method} {url}")\n\n\ndef response_json(response: requests.Response) -> Any:\n    return json.loads(response.content.decode("utf-8-sig", errors="ignore"))\n\n\ndef period_yyyymm_to_date(value: int, end_of_month: bool) -> str:\n    year = value // 100\n    month = value % 100\n    day = calendar.monthrange(year, month)[1] if end_of_month else 1\n    return f"{year:04d}-{month:02d}-{day:02d}"\n\n\ndef get_vietstock_token(session: requests.Session, symbol: str) -> tuple[str, str]:\n    page_url = f"https://finance.vietstock.vn/{symbol}/tai-chinh.htm"\n    text = request_with_retry(session, "GET", page_url).content.decode("utf-8", "ignore")\n    match = re.search(r"name=__RequestVerificationToken type=hidden value=([^ >]+)", text)\n    if not match:\n        raise RuntimeError(f"Could not find Vietstock token for {symbol}")\n    return match.group(1), page_url\n\n\ndef get_quarter_reports(session: requests.Session, symbol: str, token: str, page_url: str) -> list[dict[str, Any]]:\n    response = request_with_retry(\n        session,\n        "POST",\n        "https://finance.vietstock.vn/data/BCTT_GetListReportData",\n        headers={"Referer": page_url},\n        data={\n            "StockCode": symbol,\n            "UnitedId": "1",\n            "AuditedStatusId": "-1",\n            "Unit": "1000000",\n            "IsNamDuongLich": "false",\n            "PeriodType": "QUY",\n            "SortTimeType": "Time_ASC",\n            "__RequestVerificationToken": token,\n        },\n    )\n    payload = response_json(response)\n    return payload.get("data") or []\n\n\ndef get_report_detail_rows(\n    session: requests.Session,\n    symbol: str,\n    token: str,\n    page_url: str,\n    report: dict[str, Any],\n) -> list[dict[str, Any]]:\n    response = request_with_retry(\n        session,\n        "POST",\n        "https://finance.vietstock.vn/data/GetReportDataDetailValue_BCTT_ByReportDataIds",\n        headers={"Referer": page_url},\n        data={\n            "StockCode": symbol,\n            "Unit": "1000000",\n            "TypeCompare": "1",\n            "__RequestVerificationToken": token,\n            "listReportDataIds[0].ReportDataID": report["ReportDataID"],\n            "listReportDataIds[0].YearPeriod": report["YearPeriod"],\n            "listReportDataIds[0].ReportTermID": report["ReportTermID"],\n            "listReportDataIds[0].IsShowData": "true",\n            "listReportDataIds[0].Index": "0",\n        },\n    )\n    payload = response_json(response)\n    return payload.get("data") or []\n\n\ndef crawl_vietstock_quarterly_raw(symbol: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:\n    session = requests.Session()\n    session.trust_env = False\n    token, page_url = get_vietstock_token(session, symbol)\n    reports = get_quarter_reports(session, symbol, token, page_url)\n\n    wide_rows: list[dict[str, Any]] = []\n    long_rows: list[dict[str, Any]] = []\n    for report in reports:\n        detail_rows = get_report_detail_rows(session, symbol, token, page_url, report)\n        detail_by_norm = {int(row["ReportNormId"]): row for row in detail_rows}\n\n        period_begin = int(report["PeriodBegin"])\n        period_end = int(report["PeriodEnd"])\n        report_term_id = int(report["ReportTermID"])\n        report_quarter = report_term_id - 1\n\n        base = {\n            "ticker": symbol,\n            "report_year": int(report["YearPeriod"]),\n            "report_quarter": report_quarter,\n            "period_begin": period_begin,\n            "period_end": period_end,\n            "period_begin_date": period_yyyymm_to_date(period_begin, False),\n            "period_end_date": period_yyyymm_to_date(period_end, True),\n            "report_data_id": int(report["ReportDataID"]),\n            "report_term_id": report_term_id,\n            "audit_status": report.get("AuditStatusName", ""),\n            "source_page_url": page_url,\n            "source_pdf_url": report.get("Url", ""),\n        }\n\n        wide = dict(base)\n        for output_col, norm_id in REPORT_NORM_IDS.items():\n            raw_row = detail_by_norm.get(norm_id)\n            wide[output_col] = normalize_float(raw_row.get("Value1")) if raw_row else None\n        wide_rows.append(wide)\n\n        for raw_row in detail_rows:\n            long_rows.append(\n                {\n                    **base,\n                    "report_norm_id": int(raw_row["ReportNormId"]),\n                    "report_type_code": raw_row.get("ReportTypeCode"),\n                    **{f"value{i}": normalize_float(raw_row.get(f"Value{i}")) for i in range(1, 10)},\n                }\n            )\n        time.sleep(0.05)\n\n    quality = {\n        "ticker": symbol,\n        "quarterly_report_rows": len(wide_rows),\n        "quarterly_reportnorm_rows": len(long_rows),\n        "quarterly_first_period_end": min((row["period_end"] for row in wide_rows), default=None),\n        "quarterly_last_period_end": max((row["period_end"] for row in wide_rows), default=None),\n    }\n    return wide_rows, long_rows, quality\n', 'ocr_extractors.py': 'from __future__ import annotations\n\nimport hashlib\nimport re\nimport time\nimport unicodedata\nfrom io import StringIO\nfrom pathlib import Path\nfrom typing import Any\nfrom urllib.parse import quote\n\nimport pandas as pd\nimport requests\n\nfrom config import HF_BRANCH, HF_REPO\nfrom sources_vietstock import request_with_retry\n\n\ndef norm(text: Any) -> str:\n    return re.sub(r"\\s+", " ", str(text).lower()).strip()\n\n\ndef fold(text: Any) -> str:\n    normalized = unicodedata.normalize("NFKD", norm(text))\n    normalized = normalized.replace("đ", "d").replace("Đ", "D")\n    return "".join(ch for ch in normalized if not unicodedata.combining(ch))\n\n\ndef parse_number(value: Any) -> float | None:\n    if value is None or pd.isna(value):\n        return None\n    text = str(value).strip()\n    if not text or text in {"-", "–", "—"}:\n        return None\n    text = text.replace("\\\\(", "(").replace("\\\\)", ")").replace("\\xa0", " ")\n    match = re.search(r"\\(?-?\\d[\\d.,\\s]*\\)?", text)\n    if not match:\n        return None\n    raw = match.group(0)\n    negative = raw.startswith("(") and raw.endswith(")")\n    raw = re.sub(r"\\s+", "", raw.strip("()"))\n    if "," in raw and "." in raw:\n        if raw.rfind(",") > raw.rfind("."):\n            raw = raw.replace(".", "").replace(",", ".")\n        else:\n            raw = raw.replace(",", "")\n    elif "." in raw and "," not in raw:\n        parts = raw.split(".")\n        raw = raw if len(parts) == 2 and len(parts[1]) != 3 else raw.replace(".", "")\n    elif "," in raw and "." not in raw:\n        parts = raw.split(",")\n        raw = raw.replace(",", ".") if len(parts) == 2 and len(parts[1]) != 3 else raw.replace(",", "")\n    try:\n        val = float(raw)\n    except ValueError:\n        return None\n    return -val if negative else val\n\n\ndef row_text(row: pd.Series) -> str:\n    return " ".join(str(x) for x in row.tolist() if not pd.isna(x)).strip()\n\n\ndef first_amount_after_label(row: pd.Series, min_abs: float = 1000) -> float | None:\n    fallback = None\n    for value in row.tolist()[1:]:\n        text = str(value)\n        if "%" in text:\n            continue\n        parsed = parse_number(value)\n        if parsed is None:\n            continue\n        if fallback is None:\n            fallback = parsed\n        if abs(parsed) >= min_abs:\n            return parsed\n    return fallback\n\n\ndef numeric_values(row: pd.Series) -> list[float]:\n    values: list[float] = []\n    for value in row.tolist()[1:]:\n        parsed = parse_number(value)\n        if parsed is not None:\n            values.append(parsed)\n    return values\n\n\ndef table_to_text(df: pd.DataFrame) -> str:\n    parts = [str(col) for col in df.columns.tolist()]\n    parts.extend(row_text(row) for _, row in df.iterrows())\n    return " ".join(parts)\n\n\ndef table_amount_factor_to_million(df: pd.DataFrame) -> tuple[float, str]:\n    column_text = fold(" ".join(str(col) for col in df.columns.tolist()))\n    top_rows_text = fold(" ".join(row_text(row) for _, row in df.head(2).iterrows()))\n    folded = fold(table_to_text(df))\n    if "trieu" in column_text or "trieu" in top_rows_text or "don vi tinh trieu" in folded:\n        return 1.0, "reported_million_vnd"\n    if "don vi tinh vnd" in folded or re.search(r"\\bvnd\\b", column_text):\n        return 1 / 1_000_000, "reported_vnd_converted_to_million_vnd"\n\n    observed: list[float] = []\n    for _, row in df.iterrows():\n        observed.extend(abs(value) for value in numeric_values(row))\n    observed = [value for value in observed if value > 0]\n    if observed and max(observed) >= 10_000_000_000:\n        return 1 / 1_000_000, "inferred_vnd_converted_to_million_vnd"\n    return 1.0, "inferred_million_vnd"\n\n\ndef to_million(value: float | None, factor: float) -> float | None:\n    if value is None:\n        return None\n    return round(value * factor, 6)\n\n\ndef first_amount_after_label_million(row: pd.Series, factor: float, min_abs: float = 1000) -> float | None:\n    return to_million(first_amount_after_label(row, min_abs=min_abs), factor)\n\n\ndef hf_tree(session: requests.Session, path: str, recursive: bool = False) -> list[dict[str, Any]]:\n    encoded_path = quote(path.strip("/"))\n    url = f"https://huggingface.co/api/datasets/{HF_REPO}/tree/{HF_BRANCH}/{encoded_path}"\n    if recursive:\n        url += "?recursive=true"\n    return request_with_retry(session, "GET", url).json()\n\n\ndef hf_raw_url(path: str) -> str:\n    return f"https://huggingface.co/datasets/{HF_REPO}/resolve/{HF_BRANCH}/{quote(path)}"\n\n\ndef cache_name(path: str) -> str:\n    digest = hashlib.sha1(path.encode("utf-8")).hexdigest()[:10]\n    safe = re.sub(r"[^A-Za-z0-9_.-]+", "__", path)\n    return f"{safe}__{digest}.txt"\n\n\ndef hf_symbol_tree(session: requests.Session, symbol: str, suffix: str = "") -> list[dict[str, Any]]:\n    candidates = [f"{symbol}/{suffix}".strip("/"), f"ocr_results/{symbol}/{suffix}".strip("/")]\n    last_exc: Exception | None = None\n    for path in candidates:\n        try:\n            return hf_tree(session, path)\n        except Exception as exc:\n            last_exc = exc\n    if last_exc:\n        raise last_exc\n    return []\n\n\ndef available_hf_years(session: requests.Session, symbol: str) -> list[int]:\n    rows = hf_symbol_tree(session, symbol)\n    years: list[int] = []\n    for row in rows:\n        if row.get("type") == "directory":\n            name = Path(row["path"]).name\n            if re.fullmatch(r"\\d{4}", name):\n                years.append(int(name))\n    return sorted(set(years))\n\n\ndef choose_consolidated_hf_file(session: requests.Session, symbol: str, year: int) -> str | None:\n    candidate_root = f"ocr_results/{symbol}/{year}"\n    try:\n        rows = hf_tree(session, candidate_root, recursive=True)\n    except Exception:\n        rows = hf_symbol_tree(session, symbol, str(year))\n    files = [row["path"] for row in rows if row.get("type") == "file" and row["path"].endswith("_extracted.txt")]\n    if not files:\n        pending_dirs = [row["path"] for row in rows if row.get("type") == "directory"]\n        while pending_dirs:\n            directory = pending_dirs.pop(0)\n            for row in hf_tree(session, directory):\n                if row.get("type") == "directory":\n                    pending_dirs.append(row["path"])\n                elif row.get("type") == "file" and row["path"].endswith("_extracted.txt"):\n                    files.append(row["path"])\n    consolidated = [path for path in files if "hopnhat" in path.lower()]\n    audited = [path for path in consolidated if "kiemtoan" in path.lower()]\n    candidates = audited or consolidated or files\n    return sorted(candidates)[0] if candidates else None\n\n\ndef download_hf_text(session: requests.Session, path: str, cache_dir: Path) -> str:\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    cache_path = cache_dir / cache_name(path)\n    if cache_path.exists():\n        return cache_path.read_text(encoding="utf-8", errors="ignore")\n    response = request_with_retry(session, "GET", hf_raw_url(path))\n    cache_path.write_bytes(response.content)\n    return response.content.decode("utf-8", "ignore")\n\n\ndef parse_tables(text: str) -> list[pd.DataFrame]:\n    try:\n        return pd.read_html(StringIO(text))\n    except ValueError:\n        return []\n\n\ndef extract_direct_pct(text: str, metric: str) -> tuple[float | None, str]:\n    patterns_by_metric = {\n        "NIM": [\n            r"(?:\\bnim\\b|biên lãi ròng|net interest margin).{0,100}?(\\d{1,2}[,.]\\d{1,3})\\s*%",\n            r"(\\d{1,2}[,.]\\d{1,3})\\s*%.{0,100}?(?:\\bnim\\b|biên lãi ròng|net interest margin)",\n        ],\n        "CAR": [\n            r"(?:tỷ lệ an toàn vốn|hệ số an toàn vốn|capital adequacy ratio|\\bcar\\b).{0,120}?(\\d{1,2}[,.]\\d{1,3})\\s*%",\n            r"(\\d{1,2}[,.]\\d{1,3})\\s*%.{0,120}?(?:tỷ lệ an toàn vốn|hệ số an toàn vốn|capital adequacy ratio|\\bcar\\b)",\n        ],\n    }\n    lower = text.lower()\n    for pattern in patterns_by_metric[metric]:\n        match = re.search(pattern, lower, flags=re.IGNORECASE | re.DOTALL)\n        if match:\n            value = parse_number(match.group(1))\n            if value is not None:\n                return round(value, 6), f"direct OCR text match around {metric}"\n    return None, "not_found"\n\n\ndef extract_npl_components(tables: list[pd.DataFrame]) -> dict[str, Any]:\n    result = {\n        "npl_group3_million_vnd": None,\n        "npl_group4_million_vnd": None,\n        "npl_group5_million_vnd": None,\n        "npl_denominator_customer_loans_million_vnd": None,\n        "npl_component_method": "not_found",\n    }\n    candidates: list[dict[str, Any]] = []\n    classification_labels = [\n        "no du tieu chuan",\n        "no can chu y",\n        "no duoi tieu chuan",\n        "no nghi ngo",\n        "no co kha nang mat von",\n    ]\n    for table_index, df in enumerate(tables):\n        factor, unit_note = table_amount_factor_to_million(df)\n        labels = [fold(row_text(row)) for _, row in df.iterrows()]\n        joined = " ".join(labels)\n        if "ty le du phong" in joined or "phan loai no theo phuong phap" in joined:\n            continue\n        required = ["no duoi tieu chuan", "no nghi ngo", "no co kha nang mat von"]\n        if not all(any(label in text for text in labels) for label in required):\n            continue\n\n        values: dict[str, float | None] = {}\n        classification_components: list[float] = []\n        denominator = None\n        for _, row in df.iterrows():\n            text = fold(row_text(row))\n            val = first_amount_after_label(row)\n            label_hits = sum(1 for label in classification_labels if label in text)\n            if label_hits > 1 and "tong cong" not in text:\n                continue\n            if val is not None and label_hits == 1:\n                classification_components.append(val)\n            if "no duoi tieu chuan" in text:\n                values["group3"] = val\n            elif "no nghi ngo" in text:\n                values["group4"] = val\n            elif "no co kha nang mat von" in text:\n                values["group5"] = val\n            elif "tong cong" in text:\n                denominator = val or denominator\n\n        component_denominator = sum(classification_components) if classification_components else None\n        denominator_note = "reported total row"\n        if component_denominator and component_denominator > 0:\n            if denominator is None:\n                denominator = component_denominator\n                denominator_note = "sum of classification rows"\n            elif denominator <= 0 or abs(denominator - component_denominator) / component_denominator > 0.1:\n                denominator = component_denominator\n                denominator_note = "sum of classification rows; reported total row rejected as OCR outlier"\n        if denominator is None:\n            numeric_row_totals = [first_amount_after_label(row) for _, row in df.iterrows()]\n            numeric_row_totals = [x for x in numeric_row_totals if x is not None and abs(x) >= 1000]\n            denominator = numeric_row_totals[-1] if numeric_row_totals else None\n            denominator_note = "last numeric row fallback"\n\n        has_all_npl_components = all(values.get(key) is not None for key in ["group3", "group4", "group5"])\n        npl_amount = sum(values.get(key) or 0 for key in ["group3", "group4", "group5"])\n        if has_all_npl_components and denominator and npl_amount and 0 < npl_amount < denominator:\n            candidates.append(\n                {\n                    "npl_group3_million_vnd": to_million(values.get("group3"), factor),\n                    "npl_group4_million_vnd": to_million(values.get("group4"), factor),\n                    "npl_group5_million_vnd": to_million(values.get("group5"), factor),\n                    "npl_denominator_customer_loans_million_vnd": to_million(denominator, factor),\n                    "npl_component_method": (\n                        f"OCR table {table_index}: group 3/4/5 debt table; "\n                        f"denominator from {denominator_note}; "\n                        f"amounts standardized to million VND from {unit_note}"\n                    ),\n                }\n            )\n    if candidates:\n        result.update(max(candidates, key=lambda item: item["npl_denominator_customer_loans_million_vnd"]))\n    return result\n\n\ndef extract_loan_loss_reserve(tables: list[pd.DataFrame]) -> dict[str, Any]:\n    for table_index, df in enumerate(tables):\n        factor, unit_note = table_amount_factor_to_million(df)\n        for _, row in df.iterrows():\n            text = fold(row_text(row))\n            if "du phong rui ro cho vay khach hang" in text:\n                val = first_amount_after_label(row)\n                if val is not None:\n                    return {\n                        "loan_loss_reserve_million_vnd": abs(to_million(val, factor) or 0),\n                        "loan_loss_reserve_method": (\n                            f"OCR table {table_index}: loan loss reserve; "\n                            f"amounts standardized to million VND from {unit_note}"\n                        ),\n                    }\n    return {"loan_loss_reserve_million_vnd": None, "loan_loss_reserve_method": "not_found"}\n\n\ndef html_table_matches(text: str) -> list[re.Match[str]]:\n    return list(re.finditer(r"<table\\b.*?</table>", text, flags=re.IGNORECASE | re.DOTALL))\n\n\ndef table_prior_context(text: str, table_index: int, window: int = 700) -> str:\n    matches = html_table_matches(text)\n    if table_index >= len(matches):\n        return ""\n    start = matches[table_index].start()\n    return text[max(0, start - window) : start]\n\n\ndef is_detail_breakdown_row(text: str) -> bool:\n    return text.startswith("-") or text.startswith("+") or " bang " in text or "bang ngoai te" in text\n\n\ndef is_customer_deposit_total_label(text: str) -> bool:\n    if "tien gui cua khach hang" not in text:\n        return False\n    rejected = [\n        "va cac tctd",\n        "va cac to chuc tin dung",\n        "tien gui cua khach hang bang",\n        "ty le",\n        "lai suat",\n        "thuyet minh",\n        "phan tich",\n    ]\n    return not any(phrase in text for phrase in rejected)\n\n\ndef find_customer_deposits_from_balance_sheet(tables: list[pd.DataFrame]) -> tuple[float | None, str]:\n    candidates: list[tuple[float, str]] = []\n    for table_index, df in enumerate(tables):\n        factor, unit_note = table_amount_factor_to_million(df)\n        for _, row in df.iterrows():\n            text = fold(row_text(row))\n            if not is_customer_deposit_total_label(text):\n                continue\n            val_million = first_amount_after_label_million(row, factor)\n            if val_million is not None and val_million > 0:\n                candidates.append((val_million, f"balance-sheet row in OCR table {table_index}; {unit_note}"))\n    if not candidates:\n        return None, "not_found"\n    return max(candidates, key=lambda item: item[0])\n\n\ndef extract_customer_deposit_note_total(df: pd.DataFrame, factor: float) -> float | None:\n    totals: list[float] = []\n    for _, row in df.iterrows():\n        text = fold(row_text(row))\n        val_million = first_amount_after_label_million(row, factor)\n        if val_million is None or val_million <= 0:\n            continue\n        first_cell = fold(row.iloc[0]) if len(row.tolist()) else ""\n        numeric_only_row = bool(re.fullmatch(r"[-\\d.,\\s]+", text))\n        if "tong cong" in text or first_cell in {"", "nan", "none"} or numeric_only_row:\n            totals.append(val_million)\n    return max(totals) if totals else None\n\n\ndef extract_demand_from_customer_deposit_note(df: pd.DataFrame, factor: float) -> float | None:\n    demand_total = 0.0\n    count = 0\n    for _, row in df.iterrows():\n        text = fold(row_text(row))\n        if is_detail_breakdown_row(text):\n            continue\n        is_demand = text.startswith("tien gui khong ky han") or text.startswith("tien gui tiet kiem khong ky han")\n        if not is_demand:\n            continue\n        val_million = first_amount_after_label_million(row, factor)\n        if val_million is not None and val_million > 0:\n            demand_total += val_million\n            count += 1\n    return round(demand_total, 6) if count else None\n\n\ndef extract_casa_components(text: str, tables: list[pd.DataFrame]) -> dict[str, Any]:\n    result = {\n        "demand_deposits_million_vnd": None,\n        "customer_deposits_million_vnd": None,\n        "casa_component_method": "not_found",\n    }\n    customer_total, total_method = find_customer_deposits_from_balance_sheet(tables)\n    candidates: list[dict[str, Any]] = []\n\n    for table_index, df in enumerate(tables):\n        factor, unit_note = table_amount_factor_to_million(df)\n        table_folded = fold(table_to_text(df))\n        context_folded = fold(table_prior_context(text, table_index))\n        if "tien gui khong ky han" not in table_folded:\n            continue\n        if "tien gui cua khach hang" not in context_folded and "tien gui cua khach hang" not in table_folded:\n            continue\n        if (\n            "tien gui cua cac to chuc tin dung" in context_folded\n            or "tien gui cua tctd" in context_folded\n            or "tien gui cac to chuc tin dung" in context_folded\n            or "%/nam" in table_folded\n            or "lai suat" in table_folded\n            or "ben lien quan" in table_folded\n            or "co dong" in table_folded\n        ):\n            continue\n\n        demand = extract_demand_from_customer_deposit_note(df, factor)\n        note_total = extract_customer_deposit_note_total(df, factor)\n        total = customer_total or note_total\n        total_source = total_method if customer_total is not None else f"note total in OCR table {table_index}; {unit_note}"\n        if demand is None or total is None:\n            continue\n        if demand <= 0 or total <= 0 or demand > total * 1.001:\n            continue\n\n        candidates.append(\n            {\n                "demand_deposits_million_vnd": demand,\n                "customer_deposits_million_vnd": total,\n                "casa_component_method": (\n                    f"OCR table {table_index}: non-term customer deposits; "\n                    f"customer total from {total_source}; amounts standardized to million VND from {unit_note}"\n                ),\n            }\n        )\n\n    if candidates:\n        result.update(max(candidates, key=lambda item: item["customer_deposits_million_vnd"]))\n    elif customer_total is not None:\n        result.update(\n            {\n                "customer_deposits_million_vnd": customer_total,\n                "casa_component_method": f"customer deposit total only from {total_method}; demand deposits not found",\n            }\n        )\n    return result\n\n\ndef extract_annual_ocr_raw(symbol: str, year: int, path: str, text: str) -> dict[str, Any]:\n    tables = parse_tables(text)\n    direct_nim, nim_method = extract_direct_pct(text, "NIM")\n    direct_car, car_method = extract_direct_pct(text, "CAR")\n    row = {\n        "ticker": symbol,\n        "report_year": year,\n        "source_hf_path": path,\n        "tables_found": len(tables),\n        "direct_NIM_pct": direct_nim,\n        "direct_NIM_method": nim_method,\n        "direct_CAR_pct": direct_car,\n        "direct_CAR_method": car_method,\n    }\n    row.update(extract_npl_components(tables))\n    row.update(extract_loan_loss_reserve(tables))\n    row.update(extract_casa_components(text, tables))\n    return row\n\n\ndef crawl_annual_ocr_raw(symbols: list[str], cache_dir: Path) -> tuple[pd.DataFrame, list[dict[str, str]]]:\n    session = requests.Session()\n    session.trust_env = False\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    rows: list[dict[str, Any]] = []\n    errors: list[dict[str, str]] = []\n    for symbol in symbols:\n        try:\n            years = available_hf_years(session, symbol)\n        except Exception as exc:\n            errors.append({"ticker": symbol, "year": "", "error": repr(exc)})\n            continue\n        for year in years:\n            print(f"  HF OCR {symbol} {year}...", flush=True)\n            try:\n                path = choose_consolidated_hf_file(session, symbol, year)\n                if path is None:\n                    errors.append({"ticker": symbol, "year": str(year), "error": "no_extracted_txt_found"})\n                    continue\n                text = download_hf_text(session, path, cache_dir)\n                rows.append(extract_annual_ocr_raw(symbol, year, path, text))\n            except Exception as exc:\n                errors.append({"ticker": symbol, "year": str(year), "error": repr(exc)})\n            time.sleep(0.05)\n    return pd.DataFrame(rows), errors\n', 'quality_checks.py': 'from __future__ import annotations\n\nimport json\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _read_csv(path: Path) -> pd.DataFrame:\n    return pd.read_csv(path, encoding="utf-8-sig")\n\n\ndef run_quality_checks(outdir: Path, end_date: str) -> dict[str, Any]:\n    data = outdir / "data"\n    end_ts = pd.Timestamp(end_date)\n\n    company = _read_csv(data / "company_raw.csv")\n    daily = _read_csv(data / "daily_market_raw.csv")\n    quarterly = _read_csv(data / "quarterly_fundamental_raw.csv")\n    qlong = _read_csv(data / "quarterly_reportnorm_raw_long.csv")\n    annual = _read_csv(data / "annual_ocr_raw.csv")\n\n    for col in ["open", "high", "low", "close_vnd", "volume"]:\n        daily[col] = pd.to_numeric(daily[col], errors="coerce")\n    daily["trade_date_dt"] = pd.to_datetime(daily["trade_date"], errors="coerce")\n    ohlc_bad = (\n        (daily["high"] < daily[["open", "low", "close_vnd"]].max(axis=1))\n        | (daily["low"] > daily[["open", "high", "close_vnd"]].min(axis=1))\n    )\n\n    for col in [c for c in quarterly.columns if c.endswith("_million_vnd") or c == "roe_pct"]:\n        quarterly[col] = pd.to_numeric(quarterly[col], errors="coerce")\n    quarterly["period_end_date_dt"] = pd.to_datetime(quarterly["period_end_date"], errors="coerce")\n\n    for col in [c for c in annual.columns if c.endswith("_million_vnd") or c.endswith("_pct") or c == "tables_found"]:\n        annual[col] = pd.to_numeric(annual[col], errors="coerce")\n    npl_sum = annual[["npl_group3_million_vnd", "npl_group4_million_vnd", "npl_group5_million_vnd"]].fillna(0).sum(axis=1)\n\n    q4 = quarterly[quarterly["report_quarter"] == 4].copy()\n    merged = annual.merge(\n        q4[["ticker", "report_year", "customer_deposits_million_vnd", "customer_loans_million_vnd"]],\n        on=["ticker", "report_year"],\n        how="left",\n        suffixes=("_ocr", "_q4"),\n    )\n    deposit_ratio = (merged["customer_deposits_million_vnd_ocr"] / merged["customer_deposits_million_vnd_q4"]).replace(\n        [np.inf, -np.inf], np.nan\n    )\n    loan_ratio = (merged["npl_denominator_customer_loans_million_vnd"] / merged["customer_loans_million_vnd"]).replace(\n        [np.inf, -np.inf], np.nan\n    )\n    loan_bad = merged[loan_ratio.notna() & ((loan_ratio < 0.8) | (loan_ratio > 1.2))].copy()\n    loan_bad["ratio"] = loan_ratio.loc[loan_bad.index]\n\n    company["listing_date_dt"] = pd.to_datetime(company["listing_date"], errors="coerce")\n    first_daily = daily.groupby("ticker")["trade_date_dt"].min().rename("daily_first_date").reset_index()\n    listing_check = company.merge(first_daily, on="ticker", how="left")\n    listing_check["listing_minus_daily_first_days"] = (\n        listing_check["listing_date_dt"] - listing_check["daily_first_date"]\n    ).dt.days\n\n    listing_problem_rows = listing_check[listing_check["listing_minus_daily_first_days"] > 10][\n        ["ticker", "listing_date", "daily_first_date", "listing_minus_daily_first_days"]\n    ].copy()\n    if not listing_problem_rows.empty:\n        listing_problem_rows["daily_first_date"] = listing_problem_rows["daily_first_date"].dt.strftime("%Y-%m-%d")\n\n    return {\n        "generated_at": datetime.now().isoformat(timespec="seconds"),\n        "hard_blockers": {\n            "daily_duplicate_ticker_date": int(daily.duplicated(["ticker", "trade_date"]).sum()),\n            "daily_null_required_cells": int(\n                daily[["trade_date", "ticker", "open", "high", "low", "close_vnd", "volume"]].isna().sum().sum()\n            ),\n            "daily_bad_date_parse": int(daily["trade_date_dt"].isna().sum()),\n            "daily_dates_after_end_date": int((daily["trade_date_dt"] > end_ts).sum()),\n            "daily_negative_or_zero_price_rows": int((daily[["open", "high", "low", "close_vnd"]] <= 0).any(axis=1).sum()),\n            "daily_negative_volume": int((daily["volume"] < 0).sum()),\n            "company_duplicate_ticker": int(company.duplicated(["ticker"]).sum()),\n            "quarterly_duplicate_ticker_period_end": int(quarterly.duplicated(["ticker", "period_end"]).sum()),\n            "quarterly_period_end_after_end_date": int((quarterly["period_end_date_dt"] > end_ts).sum()),\n            "quarterly_long_duplicate_full_business_key": int(\n                qlong.duplicated(["ticker", "period_end", "report_data_id", "report_type_code", "report_norm_id"]).sum()\n            ),\n            "annual_duplicate_ticker_year": int(annual.duplicated(["ticker", "report_year"]).sum()),\n            "annual_negative_monetary_cells": int((annual[[c for c in annual.columns if c.endswith("_million_vnd")]] < 0).sum().sum()),\n            "annual_customer_deposits_overflow_gt_1e12_million": int(\n                (annual["customer_deposits_million_vnd"] > 1_000_000_000_000).sum()\n            ),\n            "annual_demand_gt_customer_deposits": int(\n                (\n                    annual["demand_deposits_million_vnd"].notna()\n                    & annual["customer_deposits_million_vnd"].notna()\n                    & (annual["demand_deposits_million_vnd"] > annual["customer_deposits_million_vnd"])\n                ).sum()\n            ),\n            "annual_npl_sum_gt_denominator": int(\n                (\n                    (npl_sum > 0)\n                    & annual["npl_denominator_customer_loans_million_vnd"].notna()\n                    & (npl_sum > annual["npl_denominator_customer_loans_million_vnd"])\n                ).sum()\n            ),\n            "annual_vs_q4_deposit_mismatch_gt_20pct": int(\n                (deposit_ratio.notna() & ((deposit_ratio < 0.8) | (deposit_ratio > 1.2))).sum()\n            ),\n        },\n        "known_data_quality_notes": {\n            "daily_ohlc_rule_violations_source_returned": int(ohlc_bad.sum()),\n            "quarterly_customer_loans_non_positive_source_returned": int((quarterly["customer_loans_million_vnd"] <= 0).sum()),\n            "quarterly_customer_deposits_non_positive_source_returned": int(\n                (quarterly["customer_deposits_million_vnd"] <= 0).sum()\n            ),\n            "annual_vs_q4_loan_denominator_mismatch_gt_20pct": int(\n                (loan_ratio.notna() & ((loan_ratio < 0.8) | (loan_ratio > 1.2))).sum()\n            ),\n            "annual_vs_q4_loan_denominator_mismatch_rows": loan_bad[\n                [\n                    "ticker",\n                    "report_year",\n                    "npl_denominator_customer_loans_million_vnd",\n                    "customer_loans_million_vnd",\n                    "ratio",\n                ]\n            ]\n            .sort_values(["ticker", "report_year"])\n            .to_dict("records"),\n            "annual_demand_deposits_nulls": int(annual["demand_deposits_million_vnd"].isna().sum()),\n            "annual_direct_nim_non_null": int(annual["direct_NIM_pct"].notna().sum()),\n            "annual_direct_car_non_null": int(annual["direct_CAR_pct"].notna().sum()),\n            "company_listing_date_after_first_daily_gt_10d_rows": listing_problem_rows.to_dict("records"),\n        },\n        "row_counts": {\n            "company_raw": int(len(company)),\n            "daily_market_raw": int(len(daily)),\n            "quarterly_fundamental_raw": int(len(quarterly)),\n            "quarterly_reportnorm_raw_long": int(len(qlong)),\n            "annual_ocr_raw": int(len(annual)),\n        },\n    }\n\n\ndef write_quality_check(path: Path, quality: dict[str, Any]) -> None:\n    path.write_text(json.dumps(quality, ensure_ascii=False, indent=2), encoding="utf-8")\n', 'crawl_bank_raw_for_sql.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport time\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\nfrom config import DEFAULT_OUTDIR, DAILY_START, HF_REPO, REPORT_NORM_NOTES, SYMBOLS, disable_proxy_env\nfrom ocr_extractors import crawl_annual_ocr_raw\nfrom quality_checks import run_quality_checks, write_quality_check\nfrom sources_vietstock import crawl_vietstock_quarterly_raw\nfrom sources_vnstock import fetch_company_raw, fetch_daily_raw\n\n\nVNSTOCK_REQUEST_DELAY_SECONDS = 3.5\n\n\ndef write_source_method(path: Path, end_date: str) -> None:\n    source_method = {\n        "generated_at": datetime.now().isoformat(timespec="seconds"),\n        "daily": {\n            "source": "vnstock Quote(source=\'KBS\').history",\n            "start": DAILY_START,\n            "end": end_date,\n            "output": "data/daily_market_raw.csv",\n            "note": "No market features are calculated in Python. Rows are deduplicated by ticker + trade_date.",\n        },\n        "company": {\n            "source": "vnstock Company(source=\'KBS\').overview",\n            "output": "data/company_raw.csv",\n            "note": "Python keeps listing/share/profile fields. SQL should treat listing_date carefully because some tickers have transfer-listing dates.",\n        },\n        "quarterly_fundamental": {\n            "token_page": "https://finance.vietstock.vn/{ticker}/tai-chinh.htm",\n            "report_list_endpoint": "https://finance.vietstock.vn/data/BCTT_GetListReportData",\n            "report_detail_endpoint": "https://finance.vietstock.vn/data/GetReportDataDetailValue_BCTT_ByReportDataIds",\n            "period_type": "QUY",\n            "unit": "1000000",\n            "outputs": ["data/quarterly_fundamental_raw.csv", "data/quarterly_reportnorm_raw_long.csv"],\n            "note": "Wide file keeps selected ReportNormId values. Long file keeps all returned ReportNormId rows.",\n        },\n        "annual_ocr": {\n            "source": f"https://huggingface.co/datasets/{HF_REPO}",\n            "api_path": "ocr_results/{ticker}/{year}",\n            "output": "data/annual_ocr_raw.csv",\n            "note": "Python parses report components only, standardizes monetary OCR components to million VND, and avoids guessing when OCR table context is unclear.",\n        },\n        "code_structure": {\n            "config.py": "Ticker list, constants, ReportNorm mapping.",\n            "sources_vnstock.py": "Daily OHLCV and company overview crawlers.",\n            "sources_vietstock.py": "Quarterly Vietstock crawler.",\n            "ocr_extractors.py": "Annual Hugging Face OCR parser for NPL, LLR, CASA, direct NIM/CAR search.",\n            "quality_checks.py": "Post-crawl audit checks.",\n            "crawl_bank_raw_for_sql.py": "Main orchestration script.",\n        },\n    }\n    path.write_text(json.dumps(source_method, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef write_data_dictionary(path: Path) -> None:\n    text = """# Raw Crawl Data Dictionary for SQL Workflow\n\nThis is the modular raw crawler version.\n\n```text\nPython crawl/parse raw source fields\n-> PostgreSQL import raw schema\n-> SQL clean/transform/feature engineering/target labeling\n-> export final master CSV\n-> RapidMiner train/evaluate\n```\n\n## Code Method\n\n| File | Responsibility |\n|---|---|\n| config.py | Constants, ticker list, ReportNormId mapping. |\n| sources_vnstock.py | Daily OHLCV and company overview from vnstock/KBS. |\n| sources_vietstock.py | Quarterly Vietstock API crawl, wide and long ReportNorm tables. |\n| ocr_extractors.py | Annual OCR parser for NPL, LLR, CASA components. |\n| quality_checks.py | Post-crawl validation and known data-quality notes. |\n| crawl_bank_raw_for_sql.py | Main orchestration and CSV export. |\n\n## data/company_raw.csv\n\n| Column | Meaning |\n|---|---|\n| ticker | Bank ticker. |\n| listing_date_raw | Listing date returned by vnstock. |\n| listing_date | Parsed YYYY-MM-DD date. |\n| outstanding_shares | Shares outstanding. |\n| exchange | Exchange from source. |\n| as_of_date | Source as-of date if available. |\n\nImportant: `listing_date` may be a transfer-listing/current-exchange date for some tickers. For `years_listed`, SQL should either use a verified listing date table or use first available daily trade date as a conservative proxy.\n\n## data/daily_market_raw.csv\n\n| Column | Meaning |\n|---|---|\n| trade_date | Trading date. |\n| ticker | Bank ticker. |\n| open, high, low, close_vnd | Source OHLC prices. |\n| volume | Trading volume. |\n\nPython removes duplicate `ticker + trade_date` rows. SQL should calculate daily returns, rolling returns, volatility, liquidity, and future labels.\n\n## data/quarterly_fundamental_raw.csv\n\nWide quarterly Vietstock ReportNorm table. Monetary columns are in million VND.\n\nImportant columns:\n\n```text\nroe_pct\noperating_profit_before_provision_million_vnd\ntotal_assets_million_vnd\ntotal_liabilities_and_equity_million_vnd\ncustomer_loans_million_vnd\ncustomer_deposits_million_vnd\nequity_million_vnd\nnet_income_million_vnd\nnet_interest_income_million_vnd\nprovision_expense_million_vnd\nreportnorm_4310_million_vnd ... reportnorm_4324_million_vnd\n```\n\nSQL should calculate NIM_proxy, debt_to_equity, current_ratio, loan_growth_yoy, deposit_growth_yoy, and other ML features.\n\n## data/quarterly_reportnorm_raw_long.csv\n\nLong audit table with every returned ReportNormId row and `value1` ... `value9`. Use it to trace missing metrics or add new metrics later.\n\n## data/annual_ocr_raw.csv\n\nAnnual OCR parsed components from Hugging Face financial statements. Monetary OCR components are standardized to million VND.\n\n| Column | Meaning |\n|---|---|\n| npl_group3_million_vnd | Group 3 debt. |\n| npl_group4_million_vnd | Group 4 debt. |\n| npl_group5_million_vnd | Group 5 debt. |\n| npl_denominator_customer_loans_million_vnd | Loan denominator for NPL ratio. |\n| loan_loss_reserve_million_vnd | Loan loss reserve for LLR. |\n| demand_deposits_million_vnd | Demand/non-term customer deposits when confidently found. |\n| customer_deposits_million_vnd | Customer deposit total from OCR/balance sheet. |\n| *_method | Audit note describing extraction method. |\n\nSQL should calculate:\n\n```text\nNPL_ratio = (group3 + group4 + group5) / customer_loans * 100\nLLR = loan_loss_reserve / (group3 + group4 + group5) * 100\nCASA_ratio = demand_deposits / customer_deposits * 100\n```\n\n## SQL-derived fields not present in raw CSV\n\n```text\nyears_listed\ndaily_return\nret_3m\nret_6m\nvol_20d\navg_volume_20d\nprice_momentum_score\nliquidity_score\nrisk_score_basic\nNIM_proxy\nNPL_ratio\nLLR\nCASA_ratio\ndebt_to_equity\ncurrent_ratio\noperating_cashflow_proxy_bn_vnd\nprovision_expense_bn_vnd\nloan_growth_yoy\ndeposit_growth_yoy\nfuture_return_63d\nfuture_volatility_63d\nfuture_max_drawdown_63d\nsegment_label\n```\n"""\n    path.write_text(text, encoding="utf-8")\n\n\ndef _deduplicate_final_daily(daily_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:\n    if daily_df.empty:\n        return daily_df, {"duplicate_rows_removed": 0, "conflicting_duplicate_keys": 0}\n    duplicate_mask = daily_df.duplicated(["ticker", "trade_date"], keep=False)\n    duplicate_rows = int(duplicate_mask.sum())\n    conflicts = 0\n    if duplicate_rows:\n        value_cols = ["open", "high", "low", "close_vnd", "volume"]\n        conflicts = int(\n            daily_df.loc[duplicate_mask]\n            .groupby(["ticker", "trade_date"])[value_cols]\n            .nunique(dropna=False)\n            .gt(1)\n            .any(axis=1)\n            .sum()\n        )\n        daily_df = daily_df.drop_duplicates(["ticker", "trade_date"], keep="last").reset_index(drop=True)\n    return daily_df, {"duplicate_rows_removed": duplicate_rows, "conflicting_duplicate_keys": conflicts}\n\n\ndef run(outdir: Path, symbols: list[str], end_date: str) -> None:\n    disable_proxy_env()\n    data_dir = outdir / "data"\n    doc_dir = outdir / "documentation"\n    cache_dir = doc_dir / "_hf_cache"\n    data_dir.mkdir(parents=True, exist_ok=True)\n    doc_dir.mkdir(parents=True, exist_ok=True)\n\n    daily_frames: list[pd.DataFrame] = []\n    company_rows: list[dict[str, Any]] = []\n    quarterly_wide_rows: list[dict[str, Any]] = []\n    quarterly_long_rows: list[dict[str, Any]] = []\n    per_ticker_quality: list[dict[str, Any]] = []\n    errors: list[dict[str, str]] = []\n\n    for symbol in symbols:\n        print(f"Crawling {symbol}...", flush=True)\n        try:\n            daily, daily_quality = fetch_daily_raw(symbol, end_date)\n            daily_frames.append(daily)\n            time.sleep(VNSTOCK_REQUEST_DELAY_SECONDS)\n        except Exception as exc:\n            errors.append({"ticker": symbol, "stage": "daily", "error": repr(exc)})\n            daily = pd.DataFrame()\n            daily_quality = {"source": "", "duplicate_input_rows": 0, "conflicting_duplicate_keys": 0}\n\n        try:\n            company, company_source = fetch_company_raw(symbol)\n            company_rows.append(company)\n            time.sleep(VNSTOCK_REQUEST_DELAY_SECONDS)\n        except Exception as exc:\n            errors.append({"ticker": symbol, "stage": "company", "error": repr(exc)})\n            company_source = ""\n\n        try:\n            q_wide, q_long, q_quality = crawl_vietstock_quarterly_raw(symbol)\n            quarterly_wide_rows.extend(q_wide)\n            quarterly_long_rows.extend(q_long)\n        except Exception as exc:\n            errors.append({"ticker": symbol, "stage": "quarterly_vietstock", "error": repr(exc)})\n            q_quality = {"ticker": symbol, "quarterly_report_rows": 0, "quarterly_reportnorm_rows": 0}\n\n        per_ticker_quality.append(\n            {\n                "ticker": symbol,\n                "daily_rows": int(len(daily)) if not daily.empty else 0,\n                "daily_first_date": daily["trade_date"].min() if not daily.empty else None,\n                "daily_last_date": daily["trade_date"].max() if not daily.empty else None,\n                "daily_source": daily_quality["source"],\n                "daily_duplicate_input_rows": daily_quality["duplicate_input_rows"],\n                "daily_conflicting_duplicate_keys": daily_quality["conflicting_duplicate_keys"],\n                "company_source": company_source,\n                **q_quality,\n            }\n        )\n\n    print("Crawling annual OCR raw components...", flush=True)\n    annual_ocr, ocr_errors = crawl_annual_ocr_raw(symbols, cache_dir)\n    errors.extend({"ticker": err.get("ticker", ""), "stage": "annual_ocr", **err} for err in ocr_errors)\n\n    daily_df = pd.concat(daily_frames, ignore_index=True) if daily_frames else pd.DataFrame()\n    daily_df, final_daily_dedupe = _deduplicate_final_daily(daily_df)\n    company_df = pd.DataFrame(company_rows)\n    q_wide_df = pd.DataFrame(quarterly_wide_rows)\n    q_long_df = pd.DataFrame(quarterly_long_rows)\n\n    daily_df.to_csv(data_dir / "daily_market_raw.csv", index=False, encoding="utf-8-sig")\n    company_df.to_csv(data_dir / "company_raw.csv", index=False, encoding="utf-8-sig")\n    q_wide_df.to_csv(data_dir / "quarterly_fundamental_raw.csv", index=False, encoding="utf-8-sig")\n    q_long_df.to_csv(data_dir / "quarterly_reportnorm_raw_long.csv", index=False, encoding="utf-8-sig")\n    annual_ocr.to_csv(data_dir / "annual_ocr_raw.csv", index=False, encoding="utf-8-sig")\n\n    write_source_method(doc_dir / "source_method.json", end_date)\n    write_data_dictionary(doc_dir / "DATA_DICTIONARY_RAW_FOR_SQL.md")\n\n    quality = run_quality_checks(outdir, end_date)\n    quality.update(\n        {\n            "outdir": str(outdir),\n            "symbols": symbols,\n            "end_date": end_date,\n            "notes": [\n                "This is a modular raw/parsed crawl. Python does not calculate ML features or target labels.",\n                "Feature engineering and target labeling should be done in SQL.",\n                "quarterly_reportnorm_raw_long.csv keeps all returned Vietstock ReportNormId rows for audit/future metrics.",\n            ],\n            "per_ticker": per_ticker_quality,\n            "errors": errors,\n            "report_norm_notes": REPORT_NORM_NOTES,\n            "final_daily_deduplication": final_daily_dedupe,\n        }\n    )\n    write_quality_check(doc_dir / "quality_check.json", quality)\n    print(json.dumps(quality["row_counts"], ensure_ascii=False, indent=2), flush=True)\n    print(f"Done: {outdir}", flush=True)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Modular raw Vietnamese bank data crawler for SQL workflow.")\n    parser.add_argument("--outdir", default=str(DEFAULT_OUTDIR))\n    parser.add_argument("--end-date", default=datetime.now().strftime("%Y-%m-%d"))\n    parser.add_argument("--symbols", nargs="*", default=SYMBOLS)\n    return parser.parse_args()\n\n\nif __name__ == "__main__":\n    args = parse_args()\n    run(Path(args.outdir), args.symbols, args.end_date)\n'}
for filename, source in EMBEDDED_SOURCES.items():
    (RUNTIME_SOURCE_DIR / filename).write_text(source, encoding="utf-8")
print("Embedded crawler modules prepared:", len(EMBEDDED_SOURCES))


In [ ]:
env = os.environ.copy()
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
env["HF_HOME"] = str(OUTPUT_DIR / "_cache" / "huggingface")
env["XDG_CACHE_HOME"] = str(OUTPUT_DIR / "_cache")

command = [
    sys.executable,
    str(RUNTIME_SOURCE_DIR / "crawl_bank_raw_for_sql.py"),
    "--outdir", str(OUTPUT_DIR),
    "--end-date", END_DATE,
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=RUNTIME_SOURCE_DIR, env=env, check=True)


In [ ]:
import hashlib
import json
import pandas as pd

expected_files = [
    "company_raw.csv",
    "daily_market_raw.csv",
    "quarterly_fundamental_raw.csv",
    "quarterly_reportnorm_raw_long.csv",
    "annual_ocr_raw.csv",
]
summary = []
for name in expected_files:
    path = OUTPUT_DIR / "data" / name
    if not path.is_file():
        raise FileNotFoundError(path)
    rows = len(pd.read_csv(path))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    summary.append({"file": name, "rows": rows, "sha256": digest})

summary_frame = pd.DataFrame(summary)
display(summary_frame)
(OUTPUT_DIR / "LIVE_CRAWL_SUMMARY.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Live crawl completed. Frozen raw data was not modified.")
